In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer: A Qiskit Function by Global Data Quantum


*ดู [API reference](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ให้บริการเฉพาะสำหรับผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (via IBM Quantum Platform API) Plan เท่านั้น ขณะนี้อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลง

## ภาพรวม
Quantum Portfolio Optimizer คือ Qiskit Function ที่ออกแบบมาเพื่อแก้ปัญหา dynamic portfolio optimization ซึ่งเป็นปัญหาพื้นฐานในด้านการเงินที่มุ่งปรับสัดส่วนการลงทุนเป็นระยะ ๆ ในสินทรัพย์หลายประเภท เพื่อเพิ่มผลตอบแทนและลดความเสี่ยง ด้วยการใช้เทคนิค quantum optimization ล้ำสมัย ฟังก์ชันนี้ทำให้กระบวนการง่ายขึ้น ผู้ใช้ที่ไม่มีความเชี่ยวชาญด้าน quantum computing ก็สามารถได้รับประโยชน์จากมันในการหา investment trajectory ที่ดีที่สุด เหมาะสำหรับผู้จัดการพอร์ตโฟลิโอ นักวิจัยด้าน quantitative finance และนักลงทุนทั่วไป เครื่องมือนี้รองรับการ back-testing กลยุทธ์การซื้อขายในการ optimize พอร์ตโฟลิโอ

## คำอธิบายฟังก์ชัน
Quantum Portfolio Optimizer ใช้อัลกอริทึม Variational Quantum Eigensolver (VQE) เพื่อแก้ปัญหา Quadratic Unconstrained Binary Optimization (QUBO) สำหรับ dynamic portfolio optimization ผู้ใช้เพียงแค่ระบุข้อมูลราคาสินทรัพย์และกำหนดเงื่อนไขการลงทุน จากนั้นฟังก์ชันจะรันกระบวนการ quantum optimization และคืนค่าชุด investment trajectory ที่ optimize แล้ว

กระบวนการประกอบด้วยสี่ขั้นตอนหลัก ขั้นแรก ข้อมูล input จะถูก map ไปยังปัญหาที่ compatible กับ quantum โดยสร้าง QUBO ของ dynamic portfolio optimization และแปลงเป็น quantum operator (Ising Hamiltonian) ต่อมา ปัญหา input และอัลกอริทึม VQE จะถูกปรับแต่งเพื่อรันบน quantum hardware จากนั้นรัน VQE บน quantum hardware และสุดท้ายผลลัพธ์จะถูก post-process เพื่อให้ได้ investment trajectory ที่ดีที่สุด ระบบยังมี noise-aware ([SQD](/guides/qiskit-addons-sqd)-based) post-processing เพื่อเพิ่มคุณภาพของ output ด้วย

Qiskit Function นี้อ้างอิงจาก [published manuscript](https://arxiv.org/abs/2412.19150) ของ Global Data Quantum

![Visualization of the workflow of the function](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)

# เริ่มต้นใช้งาน
ยืนยันตัวตนด้วย [API key](https://quantum.cloud.ibm.com/) และเลือก Qiskit Function ดังนี้ (โค้ดนี้สมมติว่าคุณ[บันทึก account](/guides/functions#install-qiskit-functions-catalog-client) ไว้ในเครื่องแล้ว)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## ตัวอย่าง: Dynamic portfolio optimization กับเจ็ดสินทรัพย์
ตัวอย่างนี้แสดงวิธีรัน dynamic portfolio optimization (DPO) function และปรับแต่งการตั้งค่าเพื่อประสิทธิภาพสูงสุด โดยมีขั้นตอนละเอียดสำหรับการ fine-tune พารามิเตอร์เพื่อให้ได้ผลลัพธ์ที่ต้องการ

กรณีนี้ใช้เจ็ดสินทรัพย์ สี่ time steps และสี่ resolution qubits ทำให้ต้องการ Qubit รวมทั้งหมด 112 ตัว

### 1. อ่านสินทรัพย์ที่รวมอยู่ใน portfolio
ถ้าสินทรัพย์ทั้งหมดใน portfolio ถูกเก็บไว้ในโฟลเดอร์ที่ path หนึ่ง สามารถโหลดลง `pandas.DataFrame` และแปลงเป็น object รูปแบบ `dict` ได้ด้วยฟังก์ชันต่อไปนี้

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

ในตัวอย่างนี้ใช้สินทรัพย์ [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) และ [XS2239553048](https://finance.yahoo.com/quote/XS2239553048) รูปต่อไปนี้แสดงข้อมูลที่ใช้ในตัวอย่าง โดยแสดงวิวัฒนาการของราคาปิดรายวันของสินทรัพย์ตั้งแต่วันที่ 1 มกราคม ถึง 1 กันยายน 2023

ในตัวอย่างนี้ เพื่อให้ข้อมูลสอดคล้องกันในทุกวัน เราได้เติมวันที่ไม่ใช่วันซื้อขายด้วยราคาปิดจากวันที่มีข้อมูลก่อนหน้า ทำขั้นตอนนี้เพราะสินทรัพย์ที่เลือกมาจากตลาดต่างกันที่มีวันซื้อขายต่างกัน จึงจำเป็นต้องทำให้ชุดข้อมูลมีมาตรฐานเดียวกัน

![Visualization of the historical data of the assets](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. กำหนดปัญหา
กำหนด specification ของปัญหาโดยตั้งค่าพารามิเตอร์ใน dictionary `qubo_settings`

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. กำหนดการตั้งค่า optimizer และ ansatz (ไม่บังคับ)
ตั้งค่าเฉพาะสำหรับกระบวนการ optimization ได้ตามต้องการ รวมถึงการเลือก optimizer และพารามิเตอร์ของมัน รวมถึงการระบุ primitive และการตั้งค่าต่าง ๆ

สำหรับ Tailored Ansatz ขนาด population size ที่เลือกมาจากการทดลองก่อนหน้าที่แสดงว่าค่านี้ให้ผล optimization ที่เสถียรและมีประสิทธิภาพ

สำหรับ Real Amplitudes Ansatz สามารถใช้ความสัมพันธ์เชิงเส้นระหว่าง ``population_size`` กับจำนวน Qubit ใน Circuit ได้ ตามกฎคร่าว ๆ แนะนำให้ใช้ ``population_size ~ 0.8 * n_qubits`` **ขั้นต่ำ** สำหรับ ansatz `real_amplitudes`

คาดว่า Optimized Real Amplitudes จะมีประสิทธิภาพ optimization ดีกว่า Real Amplitudes ansatz อย่างไรก็ตาม จำนวนตัวแปรที่ต้อง optimize ใน ansatz นี้เพิ่มขึ้นเร็วกว่า Real Amplitudes มาก (ดู[งานวิจัย](https://arxiv.org/pdf/2412.19150)) ดังนั้นสำหรับปัญหาขนาดใหญ่ Optimized Real Amplitudes ต้องการการรัน Circuit มากขึ้น Optimized Real Amplitudes น่าจะเหมาะกับปัญหาที่ต้องการ Qubit ไม่เกิน 100 ตัว แต่ควรระวังในการตั้งค่าพารามิเตอร์ ``population_size`` เพื่อเป็นตัวอย่างของการ scale-up ใน ``population_size`` ตารางก่อนหน้าแสดงว่าสำหรับปัญหา 84 Qubit Optimized Real Amplitudes ต้องการ ``population_size`` 120 ขณะที่ปัญหา 56 Qubit ใช้ ``population_size`` เพียง 40

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

นอกจากนี้ยังสามารถเลือก ansatz เฉพาะได้ ตัวอย่างต่อไปนี้ใช้ ansatz ``'Tailored'``

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. รันปัญหา

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. ดึงผลลัพธ์
ฟังก์ชันส่งคืน dictionary ที่มี investment trajectories เรียงจากต่ำสุดไปสูงสุดตามค่า objective function (ดูส่วน [Output](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) ใน API reference) ผลลัพธ์ชุดนี้ช่วยระบุ trajectory ที่มีต้นทุนต่ำสุดและการประเมินการลงทุนที่สอดคล้องกัน นอกจากนี้ยังเปิดโอกาสให้วิเคราะห์ trajectories ทางเลือกต่าง ๆ ทำให้เลือกได้ว่าแบบไหนตรงกับความต้องการหรือเป้าหมายของตัวเองมากที่สุด ความยืดหยุ่นนี้ทำให้ปรับแต่งการเลือกให้เหมาะกับสถานการณ์ที่หลากหลาย

เริ่มต้นด้วยการแสดงกลยุทธ์ผลลัพธ์ที่ได้ objective cost ต่ำสุดที่พบในระหว่างกระบวนการ

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


หลังจากนั้น ใช้ metadata เพื่อเข้าถึงผลลัพธ์ของกลยุทธ์ทั้งหมดที่ถูกสุ่ม จึงวิเคราะห์ trajectories ทางเลือกที่ optimizer ส่งคืนได้ต่อไป โดยอ่าน dictionary ที่เก็บไว้ใน `dpo_result['metadata']['all_samples_metrics']` ซึ่งมีทั้งข้อมูลเพิ่มเติมเกี่ยวกับกลยุทธ์ที่ดีที่สุด และรายละเอียดของกลยุทธ์ผู้สมัครอื่น ๆ ที่ถูกประเมินระหว่าง optimization

ตัวอย่างต่อไปนี้แสดงวิธีอ่านข้อมูลนี้โดยใช้ `pandas` เพื่อดึง key metrics ที่เกี่ยวข้องกับกลยุทธ์ที่ดีที่สุด ได้แก่ Restriction Deviation, Sharpe Ratio และผลตอบแทนการลงทุนที่สอดคล้องกัน